# Boston House Classification
Classify houses as high or low value based on housing features.

## Project Overview
Binary classification by binarizing house prices from a housing dataset. Uses California Housing as Boston is deprecated.

## Learning Objectives
- Convert regression to classification via binarization
- Work with housing features
- Compare classifiers on a well-known dataset

## Problem Statement
Given housing characteristics, classify whether a house is above or below the median price.

## Why This Project Matters
House price classification helps real estate agents, buyers, and lenders quickly categorize properties.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | sklearn.datasets.fetch_california_housing |
| **Target** | Price > median (binary) |
| **Rows** | ~20,640 |
| **Features** | 8 housing features |

## Dataset Source & License
California Housing dataset from sklearn. Public domain.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
MAX_ROWS = 50000

## Dataset Loading

In [4]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame
print(f'Shape: {df.shape}')
print(df.columns.tolist())
df.head()

Shape: (20640, 9)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')

# Binarize target at median
median_price = df['MedHouseVal'].median()
df['HighValue'] = (df['MedHouseVal'] >= median_price).astype(int)
TARGET = 'HighValue'
df = df.drop(columns=['MedHouseVal'])
print(f'\nMedian price threshold: {median_price:.2f}')
print(df[TARGET].value_counts())

Missing: 0
Duplicates: 0

Median price threshold: 1.80
HighValue
1    10325
0    10315
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#3498db','#e74c3c'], edgecolor='black')
axes[0,0].set_title('High vs Low Value')

for i, col in enumerate(['MedInc', 'AveRooms', 'HouseAge']):
    if col in df.columns:
        ax = axes[(i+1)//2, (i+1)%2]
        for label in [0, 1]:
            subset = df[df[TARGET] == label]
            ax.hist(subset[col].clip(upper=subset[col].quantile(0.99)), bins=30, alpha=0.5, label=f'{"High" if label else "Low"}')
        ax.set_title(col)
        ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print('Balanced by construction (median split)')

HighValue
1    10325
0    10315
Name: count, dtype: int64
Balanced by construction (median split)


## Train/Test Split

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (16512, 8), Test: (4128, 8)


## Preprocessing
All features are numeric. No missing values. Balanced classes from median split.

## Feature Engineering
Using raw housing features. Tree models handle scale differences.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

Baseline LR: Acc=0.8258  F1=0.8262


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                     
XGBClassifier              0.904           0.903949  0.964863  0.903989   
CatBoostClassifier         0.902           0.901972  0.966679  0.901996   
LGBMClassifier             0.901           0.900930  0.964375  0.900981   
ExtraTreesClassifier       0.894           0.894020  0.960839  0.894001   
RandomForestClassifier     0.878           0.877972  0.952064  0.877995   
BaggingClassifier          0.870           0.870127  0.929203  0.869954   
SVC                        0.859           0.858917  0.940406  0.858965   
CalibratedClassifierCV     0.849           0.848917  0.929033  0.848963   
AdaBoostClassifier         0.848           0.848091  0.921297  0.847976   
LinearSVC                  0.848           0.847911  0.929029  0.847957   
LogisticRegression         0.845           0.844916  0.927717  0.844962   
NuSVC                    

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9089  F1=0.9089
LightGBM: Acc=0.9099  F1=0.9099
XGBoost: Acc=0.9106  F1=0.9106


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
XGBoost   0.910610  0.910609   0.910646  0.910610
LightGBM  0.909884  0.909883   0.909909  0.909884
CatBoost  0.908915  0.908914   0.908925  0.908915

Top 3: ['XGBoost', 'LightGBM', 'CatBoost']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

XGBoost: Acc=0.9106  F1=0.9106
              precision    recall  f1-score   support

           0       0.91      0.92      0.91      2063
           1       0.91      0.91      0.91      2065

    accuracy                           0.91      4128
   macro avg       0.91      0.91      0.91      4128
weighted avg       0.91      0.91      0.91      4128

LightGBM: Acc=0.9099  F1=0.9099
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      2063
           1       0.91      0.91      0.91      2065

    accuracy                           0.91      4128
   macro avg       0.91      0.91      0.91      4128
weighted avg       0.91      0.91      0.91      4128



CatBoost: Acc=0.9089  F1=0.9089
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      2063
           1       0.91      0.91      0.91      2065

    accuracy                           0.91      4128
   macro avg       0.91      0.91      0.91      4128
weighted avg       0.91      0.91      0.91      4128



## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

Errors: 369, Error rate: 0.0894


## Interpretation & Business Insight
Median income is by far the strongest predictor of house value. Location (latitude/longitude) and house age also matter significantly.

## Limitations
- Binary split at median is arbitrary
- California-specific data
- Aggregate block-level features, not individual houses

## How to Improve
- Try 3-class split (low/mid/high)
- Add geographic features (distance to coast)
- Spatial cross-validation

## Production Considerations
- Regular retraining as housing market shifts
- Regional model variants
- Combine with current listing data

## Common Mistakes
- Not removing original price column (leakage)
- Using Boston dataset (deprecated for ethical reasons)
- Ignoring spatial autocorrelation

## Mini Challenge
1. Try binarizing at 75th percentile
2. Add latitude × longitude interaction
3. Compare with KNN classifier

## Final Summary
Classified California houses as high/low value. Median income dominates predictions. Boosting models achieve ~90%+ accuracy.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "XGBoost": {
    "Accuracy": 0.9106,
    "F1": 0.9106,
    "Precision": 0.9106,
    "Recall": 0.9106
  },
  "LightGBM": {
    "Accuracy": 0.9099,
    "F1": 0.9099,
    "Precision": 0.9099,
    "Recall": 0.9099
  },
  "CatBoost": {
    "Accuracy": 0.9089,
    "F1": 0.9089,
    "Precision": 0.9089,
    "Recall": 0.9089
  },
  "best_model": "XGBoost"
}
